# Pain Point Engine — SetFit Contrastive Fine-Tuning

This notebook trains SetFit classifiers on the labeled dataset using GPU-accelerated contrastive learning.

**Before running:** Go to Runtime → Change runtime type → select **T4 GPU**

**What this does:**
1. Uploads your `labeled.jsonl` and `test_ids.txt`
2. Trains two SetFit models (mpnet + modernbert) with contrastive fine-tuning
3. Evaluates on the same locked test set used locally
4. Exports trained models for download back to your Mac

**Classes (3-class):** `WORKFLOW_PAIN`, `TOOL_REQUEST`, `NOISE`
(`PRODUCT_COMPLAINT` was merged into `WORKFLOW_PAIN`)

In [ ]:
# Cell 1: Install dependencies (pinned for compatibility)
!pip install -q "setfit==1.1.0" "transformers==4.45.2" "sentence-transformers==3.1.1" "datasets==2.20.0" scikit-learn matplotlib

In [ ]:
# Cell 2: Upload your data files
from google.colab import files
print("Upload labeled.jsonl:")
uploaded = files.upload()
print("\nUpload test_ids.txt:")
uploaded2 = files.upload()

In [ ]:
# Cell 3: Load and split data
import json
from collections import Counter

LABELS = ["WORKFLOW_PAIN", "TOOL_REQUEST", "NOISE"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

# Load labeled data
texts, labels = [], []
with open("labeled.jsonl") as f:
    for line in f:
        row = json.loads(line)
        if row["label"] not in LABELS:
            continue  # skip any stale PRODUCT_COMPLAINT rows
        texts.append(row["text"])
        labels.append(row["label"])

print(f"Loaded {len(texts)} labeled posts")
print(f"Distribution: {Counter(labels)}")

# Load locked test indices
test_indices = [int(i) for i in open("test_ids.txt").read().splitlines() if i.strip()]
# test_ids.txt contains row indices into labeled.jsonl — filter to valid range
test_indices = [i for i in test_indices if i < len(texts)]
train_indices = [i for i in range(len(texts)) if i not in set(test_indices)]

train_texts = [texts[i] for i in train_indices]
train_labels = [labels[i] for i in train_indices]
test_texts = [texts[i] for i in test_indices]
test_labels = [labels[i] for i in test_indices]

print(f"Train: {len(train_texts)} | Test: {len(test_texts)}")
print(f"Train distribution: {Counter(train_labels)}")
print(f"Test distribution: {Counter(test_labels)}")

In [ ]:
# Cell 4: Train SetFit — mpnet backbone
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from setfit import SetFitModel, Trainer, TrainingArguments
from datasets import Dataset
import torch

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

backbone_mpnet = "sentence-transformers/paraphrase-mpnet-base-v2"

train_dataset = Dataset.from_dict({
    "text": train_texts,
    "label": [LABEL2ID[l] for l in train_labels],
})
eval_dataset = Dataset.from_dict({
    "text": test_texts,
    "label": [LABEL2ID[l] for l in test_labels],
})

model_mpnet = SetFitModel.from_pretrained(backbone_mpnet, labels=LABELS)

args = TrainingArguments(
    num_epochs=3,
    num_iterations=20,
    batch_size=8,
    seed=42,
)

trainer = Trainer(
    model=model_mpnet,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    metric="accuracy",
)

print("Training mpnet...")
trainer.train()
print("mpnet training complete!")

In [ ]:
# Cell 5: Train SetFit — modernbert backbone
import gc
import torch

# Free memory from mpnet training before loading next model
gc.collect()
torch.cuda.empty_cache()

backbone_modernbert = "nomic-ai/modernbert-embed-base"

model_modernbert = SetFitModel.from_pretrained(backbone_modernbert, labels=LABELS)

trainer2 = Trainer(
    model=model_modernbert,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    metric="accuracy",
)

print("Training modernbert...")
trainer2.train()
print("modernbert training complete!")

In [ ]:
# Cell 6: Evaluate both models
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

results = {}
for name, model in [("mpnet", model_mpnet), ("modernbert", model_modernbert)]:
    pred_ids = model.predict(test_texts)
    pred_labels = [ID2LABEL[int(i)] for i in pred_ids]

    print(f"\n{'='*60}")
    print(f"=== {name} ===")
    print(f"{'='*60}")
    report = classification_report(test_labels, pred_labels, labels=LABELS)
    print(report)

    kappa = cohen_kappa_score(test_labels, pred_labels, labels=LABELS)
    print(f"Cohen's Kappa: {kappa:.4f}")

    results[name] = {
        "pred_labels": pred_labels,
        "report": report,
        "kappa": kappa,
    }

    # Confusion matrix
    cm = confusion_matrix(test_labels, pred_labels, labels=LABELS)
    fig, ax = plt.subplots(figsize=(7, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS)
    disp.plot(ax=ax, xticks_rotation=30)
    ax.set_title(f"Confusion Matrix — {name} (SetFit fine-tuned)")
    fig.tight_layout()
    fig.savefig(f"confusion_matrix_{name}_finetuned.png", dpi=150)
    plt.show()
    plt.close(fig)

In [ ]:
# Cell 7: Save eval report
report_lines = []
for name in ["mpnet", "modernbert"]:
    report_lines.append(f"\n=== {name} (SetFit fine-tuned) ===\n")
    report_lines.append(results[name]["report"])
    report_lines.append(f"Cohen's Kappa: {results[name]['kappa']:.4f}\n")

with open("classification_report_finetuned.txt", "w") as f:
    f.write("\n".join(report_lines))

print("Eval report saved to classification_report_finetuned.txt")
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
for name in ["mpnet", "modernbert"]:
    print(f"{name}: Kappa = {results[name]['kappa']:.4f}")
    
# Check if we hit targets
best_kappa = max(r["kappa"] for r in results.values())
print(f"\nTarget: macro F1 >= 0.75, Kappa >= 0.70")
print(f"Best kappa: {best_kappa:.4f} {'PASS' if best_kappa >= 0.70 else 'FAIL — consider more labeled data'})")

In [ ]:
# Cell 8: Save models and download everything
import shutil

# Save both models
model_mpnet.save_pretrained("setfit-mpnet-finetuned")
model_modernbert.save_pretrained("setfit-modernbert-finetuned")

# Zip for download
shutil.make_archive("setfit-mpnet-finetuned", "zip", ".", "setfit-mpnet-finetuned")
shutil.make_archive("setfit-modernbert-finetuned", "zip", ".", "setfit-modernbert-finetuned")

print("Downloading models and eval files...")
files.download("setfit-mpnet-finetuned.zip")
files.download("setfit-modernbert-finetuned.zip")
files.download("classification_report_finetuned.txt")
files.download("confusion_matrix_mpnet_finetuned.png")
files.download("confusion_matrix_modernbert_finetuned.png")
print("Done! Unzip the model files into classifier/models/ on your Mac.")